In [1]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
test_dir = '/content/drive/MyDrive/lmu_adl_eval_history'
os.makedirs(test_dir, exist_ok=True)
test_file = os.path.join(test_dir, '_write_test.txt')
with open(test_file, 'w') as f:
    f.write('ok')
print(f'Write successful: {os.path.exists(test_file)}')
os.remove(test_file)


Write successful: True


# SGHMC & BAOA Evaluation: Pretrained-Prior vs Zero-Centered-Prior

Runs both **automated metrics** (BLEU / ROUGE / Perplexity) and **LLM-as-judge**
(Quality / Diversity / Relevance) on SGHMC and BAOA models with two prior configurations.

## 1. Setup

In [2]:
# Clone repo (kernel runs on Colab) and mount Drive
import os
if not os.path.isdir('/content/adl-bnn-textgen'):
    !git clone https://github.com/ssophiee/adl-bnn-textgen /content/adl-bnn-textgen
else:
    !git -C /content/adl-bnn-textgen pull --ff-only
    print('Repo updated.')

from google.colab import drive
drive.mount('/content/drive')

Cloning into '/content/adl-bnn-textgen'...
remote: Enumerating objects: 925, done.
remote: Counting objects: 100% (352/352), done.
remote: Compressing objects: 100% (230/230), done.
remote: Total 925 (delta 175), reused 287 (delta 117), pack-reused 573 (from 1)
Receiving objects: 100% (925/925), 33.82 MiB | 24.42 MiB/s, done.
Resolving deltas: 100% (481/481), done.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

# Resolve project root by looking for a known marker file
def find_project_root():
    # Check common locations
    candidates = [
        os.getcwd(),
        os.path.abspath(os.path.join(os.getcwd(), '..')),
        '/content/adl-bnn-textgen',
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, 'config.py')):
            return c
    raise FileNotFoundError('Could not find project root (looked for config.py)')

PROJECT_ROOT = find_project_root()
print(f'Project root: {PROJECT_ROOT}')
!git -C "{PROJECT_ROOT}" log --oneline -3

Project root: /content/adl-bnn-textgen
9212457 (HEAD -> main, origin/main, origin/HEAD) feat(eval): add resume/checkpoint support to LLM and auto metrics pipelines
28175f6 fix(prior): use magnitude-scaled prior std and sync global CONFIG
a76e82a docs: add explicit prior configuration section to README and report


In [4]:
# Write .env BEFORE any project imports (config.py reads it at import time)
env_content = f"""BASE_DIR={PROJECT_ROOT}
MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
META_PATH=/content/drive/MyDrive/baseline/nanogpt_meta.pkl
DATA_DIR={PROJECT_ROOT}/data/tokenized
BNN_MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
DEVICE=cuda
WANDB_AVAILABLE=false
"""

env_path = os.path.join(PROJECT_ROOT, '.env')
with open(env_path, 'w') as f:
    f.write(env_content)

print(env_content)
print(f'Environment file written to: {env_path}')

BASE_DIR=/content/adl-bnn-textgen
MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
META_PATH=/content/drive/MyDrive/baseline/nanogpt_meta.pkl
DATA_DIR=/content/adl-bnn-textgen/data/tokenized
BNN_MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
DEVICE=cuda
WANDB_AVAILABLE=false

Environment file written to: /content/adl-bnn-textgen/.env


In [5]:
!pip install -q python-dotenv posteriors evaluate rouge_score datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.2/168.2 kB 21.1 MB/s eta 0:00:00


## 2. Discover models

In [7]:
import glob, os

# --- SGHMC models ---
SGHMC_BASE = '/content/drive/MyDrive/samplers/sghmc_sampler'
sghmc_pretrained = sorted(glob.glob(f'{SGHMC_BASE}/pretrained-prior/run_*/sghmc_model.pt'))
sghmc_zero = sorted(glob.glob(f'{SGHMC_BASE}/zero-centered-prior/run_*/sghmc_model.pt'))

print(f'SGHMC pretrained-prior ({len(sghmc_pretrained)}):')
for p in sghmc_pretrained:
    print(f'  {p}')
print(f'SGHMC zero-centered-prior ({len(sghmc_zero)}):')
for p in sghmc_zero:
    print(f'  {p}')

# --- BAOA models ---
BAOA_BASE = '/content/drive/MyDrive/samplers/baoa_sampler'
baoa_pretrained = sorted(glob.glob(f'{BAOA_BASE}/pretrained-prior/run_*/baoa_model.pt'))
baoa_zero = sorted(glob.glob(f'{BAOA_BASE}/zero-centered-prior/run_*/baoa_model.pt'))

print(f'\nBAOA pretrained-prior ({len(baoa_pretrained)}):')
for p in baoa_pretrained:
    print(f'  {p}')
print(f'BAOA zero-centered-prior ({len(baoa_zero)}):')
for p in baoa_zero:
    print(f'  {p}')

# Build combined lists: models + their sampler types
all_models = sghmc_pretrained + sghmc_zero + baoa_pretrained + baoa_zero
all_sampler_types = (
    ['sghmc'] * (len(sghmc_pretrained) + len(sghmc_zero))
    + ['baoa'] * (len(baoa_pretrained) + len(baoa_zero))
)

print(f'\nTotal models: {len(all_models)}')
assert len(all_models) == 8, f'Expected 8 models, found {len(all_models)}. Check the paths above.'

SGHMC pretrained-prior (2):
  /content/drive/MyDrive/samplers/sghmc_sampler/pretrained-prior/run_20260224-230657/sghmc_model.pt
  /content/drive/MyDrive/samplers/sghmc_sampler/pretrained-prior/run_20260224-232547/sghmc_model.pt
SGHMC zero-centered-prior (2):
  /content/drive/MyDrive/samplers/sghmc_sampler/zero-centered-prior/run_20260224-231606/sghmc_model.pt
  /content/drive/MyDrive/samplers/sghmc_sampler/zero-centered-prior/run_20260224-233423/sghmc_model.pt

BAOA pretrained-prior (2):
  /content/drive/MyDrive/samplers/baoa_sampler/pretrained-prior/run_20260207-192540/baoa_model.pt
  /content/drive/MyDrive/samplers/baoa_sampler/pretrained-prior/run_20260225-101215/baoa_model.pt
BAOA zero-centered-prior (2):
  /content/drive/MyDrive/samplers/baoa_sampler/zero-centered-prior/run_20260207-232141/baoa_model.pt
  /content/drive/MyDrive/samplers/baoa_sampler/zero-centered-prior/run_20260225-102111/baoa_model.pt

Total models: 8


## 3. LLM-as-Judge Evaluation (Generation + Scoring)

In [9]:
import sys, json
sys.path.insert(0, PROJECT_ROOT)

# Load .env explicitly (cwd is /content, but .env is in PROJECT_ROOT)
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'), override=True)

# Extract prompts from the testing results file
gen_results_path = os.path.join(PROJECT_ROOT, 'results/evaluation/llm_judge/external_data/generation_results_testing.json')
with open(gen_results_path, 'r') as f:
    gen_data = json.load(f)

test_prompts = sorted(set(v['prompt'] for v in gen_data.values() if 'prompt' in v))
print(f'Extracted {len(test_prompts)} unique prompts')
for i, p in enumerate(test_prompts):
    print(f'  {i+1}. {p[:60]}...')

# Save prompts for reference
prompts_out = os.path.join(PROJECT_ROOT, 'results/evaluation/llm_judge/external_prompts.json')
with open(prompts_out, 'w') as f:
    json.dump(test_prompts, f, indent=2)

# Import the evaluation pipeline
from scripts.llm_evaluation import run_evaluation_pipeline
print(f'\nrun_evaluation_pipeline imported successfully')

Extracted 15 unique prompts
  1. And so it was, that in those days,...
  2. HAMLET: To be, or not to be, that is the question:...
  3. In such a manner, one might say,...
  4. In the beginning, there was darkness and light,...
  5. JULIET: O Romeo, Romeo! Wherefore art thou Romeo?...
  6. Love conquers all, they say, but I have found...
  7. MACBETH: Is this a dagger which I see before me,...
  8. OPHELIA: My lord, I have remembrances of yours,...
  9. Once upon a time in a kingdom far away,...
  10. ROMEO: But soft! What light through yonder window breaks?...
  11. She looked into his eyes and saw...
  12. The greatest treasure in all the world is not...
  13. The king turned to his advisor and...
  14. The old king looked upon his realm and said:...
  15. The thing that happened next was most...

run_evaluation_pipeline imported successfully


In [ ]:
# FULL RUN: 8 models × 15 prompts × 8 param combos
# Grid matches previous evaluation: temp=[0.3,0.8], top_k=[10,50], num_samples=[10,30]
# Results saved to Drive so they survive runtime disconnects

model_types = ['bayesian'] * len(all_models)

DRIVE_OUTPUT = '/content/drive/MyDrive/lmu_adl_eval_history'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Override default grid to match previous evaluation
from scripts.llm_evaluation import EvaluationConfig
EvaluationConfig.DEFAULT_TEMPERATURE = [0.3, 0.8]
EvaluationConfig.DEFAULT_TOP_K = [10, 50]
EvaluationConfig.DEFAULT_NUM_SAMPLES = [10, 30]

aggregated, full_results = run_evaluation_pipeline(
    test_prompts=test_prompts,
    model_paths=all_models,
    model_types=model_types,
    change_params=True,
    output_path=f'{DRIVE_OUTPUT}/generation_results.json',
    use_local_qwen=True,
    device='cuda',
    use_external_data=True,
)


################################################################################
# LLM Evaluation Pipeline
# Started at: 2026-03-03 08:31:47
################################################################################

Using 15 provided prompts

Loaded 1 previous results from: /content/drive/MyDrive/lmu_adl_eval_history/generation_results.json

Starting Text Generation Phase
Total generations to perform: 960
Already completed (resuming): 1
Remaining: 959
Models: 8
Prompts: 15
Parameter combinations: 8


--- Processing model 1/8: sghmc_model.pt (type: bayesian) ---

  Prompt: 'And so it was, that in those days,...' [source: unknown]
    [1/960] temp=0.3, top_k=10, samples=10... SKIP (cached)
    [2/960] temp=0.3, top_k=10, samples=30... Loaded 100 collected samples from checkpoint
number of parameters: 10.65M
Using 30 SGMCMC samples for generation
✓
Saved generation results to: /content/drive/MyDrive/lmu_adl_eval_history/generation_results.json
    [3/960] temp=0.3, top_k=50, sampl

In [24]:
# Print summary
from pathlib import Path

print('\n' + '='*80)
print('LLM-Judge Results Summary')
print('='*80)
for model_path, scores in aggregated.items():
    run_dir = Path(model_path).parent.name
    prior_type = Path(model_path).parent.parent.name
    print(f'\n{prior_type} / {run_dir}:')
    print(f'  Quality:   {scores["overall_avg_quality"]:.2f}')
    print(f'  Diversity: {scores["overall_avg_diversity"]:.2f}')
    print(f'  Relevance: {scores["overall_avg_relevance"]:.2f}')
    print(f'  Generations: {scores["total_generations"]}')


LLM-Judge Results Summary

pretrained-prior / run_20260224-230657:
  Quality:   5.50
  Diversity: 6.00
  Relevance: 7.50
  Generations: 1


## 4. Automated Metrics (BLEU / ROUGE / Perplexity)

In [ ]:
import torch
from pathlib import Path

from src.nanogpt_utils import load_model, load_tokenizer
from src.generation_utils import load_checkpoint_for_generation
from scripts.bayesian_evaluator import BayesianNanoGPTEvaluator, evaluate_bayesian_splits

META_PATH = '/content/drive/MyDrive/baseline/nanogpt_meta.pkl'
DATA_DIR  = os.path.join(PROJECT_ROOT, 'data/tokenized')
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

stoi, itos = load_tokenizer(Path(META_PATH))
print(f'Vocab size: {len(itos)}, Device: {DEVICE}')

In [ ]:
import json, traceback, time

AUTO_METRICS_PATH = f'{DRIVE_OUTPUT}/automated_metrics.json'

def _save_auto_checkpoint():
    with open(AUTO_METRICS_PATH, 'w') as f:
        json.dump(auto_results, f, indent=2, default=str)

# Auto-resume: load previous results from Drive
auto_results = {}
if os.path.exists(AUTO_METRICS_PATH):
    with open(AUTO_METRICS_PATH, 'r') as f:
        auto_results = json.load(f)
    print(f'Resumed {len(auto_results)} previous model results from Drive')

eval_config = {
    'data_dir': DATA_DIR,
    'splits': ['train', 'val'],
    'batch_size': 16,
    'max_eval_samples': 500,
    'num_text_samples': 30,
    'prompt_length': 20,
    'generation_length': 30,
    'max_tokens': None,
}

METRIC_STEPS = ['perplexity_external_gpt2', 'bleu']  # bleu step also does rouge

for model_path, sampler_type in zip(all_models, all_sampler_types):
    run_dir = Path(model_path).parent.name
    prior_type = Path(model_path).parent.parent.name
    label = f'{sampler_type}/{prior_type}/{run_dir}'

    # Check which (split, metric) pairs are still needed
    existing = auto_results.get(label, {})
    work = []
    for split in eval_config['splits']:
        split_data = existing.get(split, {})
        if isinstance(split_data, dict) and 'error' not in split_data:
            for step in METRIC_STEPS:
                if step not in split_data:
                    work.append((split, step))
        else:
            work.extend([(split, step) for step in METRIC_STEPS])

    if not work:
        print(f'\nSKIP (cached): {label}')
        continue

    print(f'\n{"="*60}')
    print(f'Evaluating: {label} ({len(work)} steps remaining)')
    print(f'{"="*60}')

    model, _ = load_model(Path('/content/drive/MyDrive/baseline/baseline_model_2k.pt'), DEVICE)
    ckpt = load_checkpoint_for_generation(model_path, device=DEVICE)
    collected_samples = ckpt.get('collected_samples', [])
    print(f'  Collected samples: {len(collected_samples)}')

    evaluator = BayesianNanoGPTEvaluator(
        model=model, stoi=stoi, itos=itos,
        sampler_type=sampler_type,
        collected_samples=collected_samples,
        device=DEVICE,
        num_posterior_samples=min(10, len(collected_samples)),
    )

    if label not in auto_results:
        auto_results[label] = {'model_path': model_path, 'prior_type': prior_type, 'sampler_type': sampler_type}

    for split in eval_config['splits']:
        if split not in auto_results[label] or not isinstance(auto_results[label].get(split), dict):
            auto_results[label][split] = {'split': split, 'sampler_type': sampler_type}

        split_data = auto_results[label][split]
        if 'error' in split_data:
            split_data = {'split': split, 'sampler_type': sampler_type}
            auto_results[label][split] = split_data

        data = evaluator.load_data(eval_config['data_dir'], split, eval_config.get('max_tokens'))
        max_ppl_batches = min(100, eval_config['max_eval_samples'] // eval_config['batch_size'])

        # Step 1: External GPT-2 perplexity
        if 'perplexity_external_gpt2' not in split_data:
            print(f'  [{split}] perplexity_gpt2...', end=' ', flush=True)
            t0 = time.time()
            try:
                ppl_ext = evaluator.calculate_perplexity_external_gpt2(data, eval_config['batch_size'], max_batches=max_ppl_batches)
                split_data['perplexity_external_gpt2'] = ppl_ext if ppl_ext is not None else 0.0
                print(f'{split_data["perplexity_external_gpt2"]:.1f} ({time.time()-t0:.0f}s)')
            except Exception as e:
                split_data['perplexity_external_gpt2'] = 0.0
                print(f'ERROR: {e}')
            _save_auto_checkpoint()

        # Step 2: Text generation + BLEU + ROUGE
        if 'bleu' not in split_data:
            print(f'  [{split}] generate + bleu/rouge...', end=' ', flush=True)
            t0 = time.time()
            try:
                refs, preds = evaluator.generate_samples_for_metrics(
                    data, eval_config['num_text_samples'],
                    eval_config['prompt_length'], eval_config['generation_length'])
                if refs and preds:
                    bleu = evaluator.calculate_bleu_score(refs, preds)
                    split_data['bleu'] = bleu if bleu is not None else 0.0
                    rouge = evaluator.calculate_rouge_score(refs, preds)
                    split_data.update(rouge)
                else:
                    split_data.update({'bleu': 0.0, 'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0})
                print(f'BLEU={split_data["bleu"]:.4f} ROUGE-2={split_data.get("rouge2",0):.4f} ({time.time()-t0:.0f}s)')
            except Exception as e:
                split_data.update({'bleu': 0.0, 'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0})
                print(f'ERROR: {e}')
            _save_auto_checkpoint()

    print(f'  Done: {label}')

print(f'\nAll {len(auto_results)} models evaluated.')

## 5. Save Results

In [ ]:
from datetime import datetime

print(f'Automated metrics already saved to: {AUTO_METRICS_PATH}')
print(f'LLM-judge results saved to: {DRIVE_OUTPUT}/generation_results*.json')
print(f'\nAll results on Drive at {datetime.now():%Y-%m-%d %H:%M:%S}')

## 6. Quick Comparison Table

In [ ]:
import pandas as pd

rows = []
for label, data in auto_results.items():
    for split in ['train', 'val']:
        m = data.get(split, {})
        if isinstance(m, dict) and 'error' not in m:
            rows.append({
                'model': label,
                'split': split,
                'bleu': m.get('bleu', 0),
                'rouge1': m.get('rouge1', 0),
                'rouge2': m.get('rouge2', 0),
                'rougeL': m.get('rougeL', 0),
                'ppl_gpt2': m.get('perplexity_external_gpt2', 0),
            })

df = pd.DataFrame(rows)
if not df.empty:
    display(df.round(4))
else:
    print('No results to display.')